<center><h1>Data Cleaning and Preprocessing</h1></center>


#### Load Data

In [1]:
# Load raw datasets for the data cleaning phase

import pandas as pd

df = pd.read_csv("../data/raw/sales_daily.csv")
df_inventory = pd.read_csv("../data/raw/inventory_snapshots.csv")
df_calendar = pd.read_csv("../data/raw/calendar.csv")

print("Sales data shape:", df.shape)
print("Inventory data shape:", df_inventory.shape)
print("Calendar data shape:", df_calendar.shape)

Sales data shape: (36550, 6)
Inventory data shape: (4800, 8)
Calendar data shape: (731, 11)


### Standardize Dates and Inspect Missing Values

#### Convert date columns to datetime format and assess missing values in the sales, inventory, and calendar datasets.

In [3]:
# Standardize date columns and inspect missing values

df["Date"] = pd.to_datetime(df["Date"])
df_inventory["Snapshot_Date"] = pd.to_datetime(df_inventory["Snapshot_Date"])
df_calendar["date"] = pd.to_datetime(df_calendar["date"])

print("Sales missing values:")
display(df.isnull().sum())

print("Inventory missing values:")
display(df_inventory.isnull().sum())

print("Calendar missing values:")
display(df_calendar.isnull().sum())

Sales missing values:


Date          0
SKU           0
Units_Sold    0
Revenue       0
Price         0
Promotion     0
dtype: int64

Inventory missing values:


Snapshot_Date      0
SKU                0
Current_Stock      0
On_Order           0
Lead_Time_Days     0
Safety_Stock       0
Reorder_Point      0
Inventory_Value    0
dtype: int64

Calendar missing values:


date                 0
year                 0
month                0
quarter              0
week                 0
day_of_week          0
is_weekend           0
season               0
holiday            723
is_holiday           0
promotion_event    656
dtype: int64

### Inspect Calendar Fields Before Cleaning

#### Examine holiday labels and promotion-event records to understand whether blank values represent dates without a holiday or promotion.

In [5]:
# Inspect blank calendar fields before deciding how to handle them

print("Holiday values for non-holiday dates:")
display(df_calendar.loc[df_calendar["is_holiday"] == 0, "holiday"].value_counts(dropna=False))

print("\nHoliday values for holiday dates:")
display(df_calendar.loc[df_calendar["is_holiday"] == 1, "holiday"].value_counts(dropna=False))

print("\nPromotion event examples:")
display(df_calendar.loc[df_calendar["promotion_event"].notna(),
                         ["date", "is_holiday", "holiday", "promotion_event"]].head(10))

Holiday values for non-holiday dates:


holiday
NaN    723
Name: count, dtype: int64


Holiday values for holiday dates:


holiday
Republic Day        2
Independence Day    2
Diwali              2
Christmas           2
Name: count, dtype: int64


Promotion event examples:


,date,is_holiday,holiday,promotion_event
61,2024-03-02,0,NaN,Seasonal Promotion
62,2024-03-03,0,NaN,Seasonal Promotion
68,2024-03-09,0,NaN,Seasonal Promotion
69,2024-03-10,0,NaN,Seasonal Promotion
75,2024-03-16,0,NaN,Seasonal Promotion
76,2024-03-17,0,NaN,Seasonal Promotion
82,2024-03-23,0,NaN,Seasonal Promotion
83,2024-03-24,0,NaN,Seasonal Promotion
89,2024-03-30,0,NaN,Seasonal Promotion
90,2024-03-31,0,NaN,Seasonal Promotion


### Clean Holiday Labels

#### Create a separate cleaned calendar DataFrame and replace missing holiday labels with No Holiday, assuming blank labels indicate dates without a holiday.

In [7]:
# Clean calendar holiday labels while preserving the original dataset

df_calendar_clean = df_calendar.copy()

df_calendar_clean["holiday"] = df_calendar_clean["holiday"].fillna("No Holiday")

print("Missing holiday values:", df_calendar_clean["holiday"].isna().sum())
print("\nHoliday categories:")
display(df_calendar_clean["holiday"].value_counts())
print("\nMissing promotion event values:", df_calendar_clean["promotion_event"].isna().sum())

Missing holiday values: 0

Holiday categories:


holiday
No Holiday          723
Republic Day          2
Independence Day      2
Diwali                2
Christmas             2
Name: count, dtype: int64


Missing promotion event values: 656


### Analyze Promotion Event Categories

#### Review promotion-event categories and their yearly distribution before handling missing promotion labels.

In [9]:
# Inspect promotion event categories and their date distribution

print("Promotion event counts:")
display(df_calendar_clean["promotion_event"].value_counts(dropna=False))

print("\nPromotion events by year:")
display(
    df_calendar_clean.groupby("year")["promotion_event"]
    .agg(
        Total_Dates="size",
        Promotion_Event_Dates="count"
    )
)

Promotion event counts:


promotion_event
NaN                   656
Seasonal Promotion     75
Name: count, dtype: int64


Promotion events by year:


,Total_Dates,Promotion_Event_Dates
year,,
2024,366,38
2025,365,37


### Handle Missing Promotion Events

#### Replace missing promotion-event labels with No Promotion, assuming blank values indicate dates without a promotion.

In [11]:
# Handle blank promotion events using an explicit project assumption

df_calendar_clean["promotion_event"] = (
    df_calendar_clean["promotion_event"].fillna("No Promotion")
)

print("Missing promotion events:", df_calendar_clean["promotion_event"].isna().sum())
display(df_calendar_clean["promotion_event"].value_counts())

Missing promotion events: 0


promotion_event
No Promotion          656
Seasonal Promotion     75
Name: count, dtype: int64

### Validate the Cleaned Calendar Dataset

#### Check the dataset dimensions, missing values, duplicate rows, and unique dates after cleaning.

In [13]:
# Validate the cleaned calendar dataset

print("Calendar shape:", df_calendar_clean.shape)
print("\nMissing values:")
display(df_calendar_clean.isnull().sum())

print("\nDuplicate rows:", df_calendar_clean.duplicated().sum())
print("Unique dates:", df_calendar_clean["date"].nunique())

Calendar shape: (731, 11)

Missing values:


date               0
year               0
month              0
quarter            0
week               0
day_of_week        0
is_weekend         0
season             0
holiday            0
is_holiday         0
promotion_event    0
dtype: int64


Duplicate rows: 0
Unique dates: 731


### Save the Cleaned Calendar Dataset

### Create the processed data directory if needed and save the cleaned calendar dataset as a CSV file.

In [15]:
# Save the cleaned calendar dataset

from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_calendar_clean.to_csv(output_dir / "calendar_clean.csv", index=False)

print("Saved:", output_dir / "calendar_clean.csv")

Saved: ..\data\processed\calendar_clean.csv


In [17]:
# Validate sales data before cleaning

print("Sales shape:", df.shape)
print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Zero-sales records:", (df["Units_Sold"] == 0).sum())
print("Negative-sales records:", (df["Units_Sold"] < 0).sum())

Sales shape: (36550, 6)

Missing values:


Date          0
SKU           0
Units_Sold    0
Revenue       0
Price         0
Promotion     0
dtype: int64


Duplicate rows: 0
Date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00
Zero-sales records: 202
Negative-sales records: 0


### Validate Revenue Consistency

#### Compare recorded revenue with the expected value calculated from units sold and price. Inspect mismatched records before deciding whether any correction is appropriate.

In [19]:
# Check whether revenue matches units sold multiplied by price

df["Expected_Revenue"] = (df["Units_Sold"] * df["Price"]).round(2)
df["Revenue_Difference"] = (df["Revenue"] - df["Expected_Revenue"]).abs().round(2)

print("Records with revenue mismatch:", (df["Revenue_Difference"] > 0.01).sum())
print("\nExamples of mismatched records:")
display(
    df.loc[df["Revenue_Difference"] > 0.01,
           ["Date", "SKU", "Units_Sold", "Price", "Revenue",
            "Expected_Revenue", "Revenue_Difference"]].head(10)
)

Records with revenue mismatch: 0

Examples of mismatched records:


,Date,SKU,Units_Sold,Price,Revenue,Expected_Revenue,Revenue_Difference


### Save the Cleaned Sales Dataset

#### Remove temporary revenue-validation columns and save the sales dataset for downstream analysis.

In [21]:
# Remove temporary validation columns and save cleaned sales data

df_sales_clean = df.drop(
    columns=["Expected_Revenue", "Revenue_Difference"]
).copy()

df_sales_clean.to_csv("../data/processed/sales_clean.csv", index=False)

print("Cleaned sales shape:", df_sales_clean.shape)
print("Saved: ../data/processed/sales_clean.csv")

Cleaned sales shape: (36550, 6)
Saved: ../data/processed/sales_clean.csv


### Validate Inventory Data Before Cleaning

#### Inspect missing values, duplicate rows, date coverage, and invalid stock quantities or inventory values.

In [23]:
# Validate inventory data before cleaning

print("Inventory shape:", df_inventory.shape)
print("\nMissing values:")
display(df_inventory.isnull().sum())

print("\nDuplicate rows:", df_inventory.duplicated().sum())
print("Date range:", df_inventory["Snapshot_Date"].min(),
      "to", df_inventory["Snapshot_Date"].max())

print("Negative current stock:",
      (df_inventory["Current_Stock"] < 0).sum())
print("Negative on-order quantity:",
      (df_inventory["On_Order"] < 0).sum())
print("Non-positive inventory value:",
      (df_inventory["Inventory_Value"] <= 0).sum())

Inventory shape: (4800, 8)

Missing values:


Snapshot_Date      0
SKU                0
Current_Stock      0
On_Order           0
Lead_Time_Days     0
Safety_Stock       0
Reorder_Point      0
Inventory_Value    0
dtype: int64


Duplicate rows: 0
Date range: 2024-01-01 00:00:00 to 2025-12-01 00:00:00
Negative current stock: 0
Negative on-order quantity: 0
Non-positive inventory value: 0


In [25]:
# Save the validated inventory dataset

df_inventory_clean = df_inventory.copy()

df_inventory_clean.to_csv(
    "../data/processed/inventory_clean.csv",
    index=False
)

print("Cleaned inventory shape:", df_inventory_clean.shape)
print("Saved: ../data/processed/inventory_clean.csv")

Cleaned inventory shape: (4800, 8)
Saved: ../data/processed/inventory_clean.csv


### Validate Saved Processed Datasets

#### Reload the sales, inventory, and calendar CSV files and verify their shapes, missing values, and duplicate rows.

In [27]:
# Validate all cleaned datasets saved in the processed folder

from pathlib import Path

processed_dir = Path("../data/processed")

for file_name in ["sales_clean.csv", "inventory_clean.csv", "calendar_clean.csv"]:
    file_path = processed_dir / file_name
    check_df = pd.read_csv(file_path)

    print(f"\n{file_name}")
    print("Shape:", check_df.shape)
    print("Missing values:", check_df.isnull().sum().sum())
    print("Duplicate rows:", check_df.duplicated().sum())


sales_clean.csv
Shape: (36550, 6)
Missing values: 0
Duplicate rows: 0

inventory_clean.csv
Shape: (4800, 8)
Missing values: 0
Duplicate rows: 0

calendar_clean.csv
Shape: (731, 11)
Missing values: 0
Duplicate rows: 0


### Merge Sales Data with Calendar Features

#### Join the cleaned sales data with calendar attributes using the date column. Validate the many-to-one relationship to avoid accidentally multiplying sales rows

In [29]:
# Merge cleaned sales data with calendar features by date

df_sales_clean = pd.read_csv("../data/processed/sales_clean.csv")
df_calendar_clean = pd.read_csv("../data/processed/calendar_clean.csv")

df_sales_clean["Date"] = pd.to_datetime(df_sales_clean["Date"])
df_calendar_clean["date"] = pd.to_datetime(df_calendar_clean["date"])

df_sales_calendar = df_sales_clean.merge(
    df_calendar_clean,
    left_on="Date",
    right_on="date",
    how="left",
    validate="many_to_one"
)

print("Merged dataset shape:", df_sales_calendar.shape)
print("Missing values:", df_sales_calendar.isnull().sum().sum())
print("Sales rows before merge:", len(df_sales_clean))
print("Sales rows after merge:", len(df_sales_calendar))

display(df_sales_calendar.head())

Merged dataset shape: (36550, 17)
Missing values: 0
Sales rows before merge: 36550
Sales rows after merge: 36550


,Date,SKU,Units_Sold,Revenue,Price,Promotion,date,year,month,quarter,week,day_of_week,is_weekend,season,holiday,is_holiday,promotion_event
0,2024-01-01,SKU001,5,18320.25,3664.05,0,2024-01-01,2024,1,Q1,1,Monday,0,Winter,No Holiday,0,No Promotion
1,2024-01-01,SKU002,15,57085.35,3805.69,0,2024-01-01,2024,1,Q1,1,Monday,0,Winter,No Holiday,0,No Promotion
2,2024-01-01,SKU003,5,40391.15,8078.23,0,2024-01-01,2024,1,Q1,1,Monday,0,Winter,No Holiday,0,No Promotion
3,2024-01-01,SKU004,5,34307.85,6861.57,0,2024-01-01,2024,1,Q1,1,Monday,0,Winter,No Holiday,0,No Promotion
4,2024-01-01,SKU005,12,113918.64,9493.22,0,2024-01-01,2024,1,Q1,1,Monday,0,Winter,No Holiday,0,No Promotion


### Save the Integrated Sales-Calendar Dataset

#### Remove the redundant calendar date column and export the integrated dataset for downstream feature engineering and forecasting.

In [31]:
# Finalize and save the merged sales-calendar dataset

df_sales_calendar = df_sales_calendar.drop(columns=["date"])

output_path = "../data/processed/sales_calendar_clean.csv"
df_sales_calendar.to_csv(output_path, index=False)

print("File saved successfully:", output_path)
print("Final dataset shape:", df_sales_calendar.shape)
print("Missing values:", df_sales_calendar.isnull().sum().sum())
print("Duplicate rows:", df_sales_calendar.duplicated().sum())
print("Columns:", df_sales_calendar.columns.tolist())

File saved successfully: ../data/processed/sales_calendar_clean.csv
Final dataset shape: (36550, 16)
Missing values: 0
Duplicate rows: 0
Columns: ['Date', 'SKU', 'Units_Sold', 'Revenue', 'Price', 'Promotion', 'year', 'month', 'quarter', 'week', 'day_of_week', 'is_weekend', 'season', 'holiday', 'is_holiday', 'promotion_event']


### Final Validation of Processed Files

#### Reload all processed datasets and verify their dimensions, missing values, and duplicate rows before using them in subsequent notebooks.

In [33]:

# Validate all processed datasets saved on disk

from pathlib import Path
import pandas as pd

processed_dir = Path("../data/processed")

for file_name in [
    "sales_clean.csv",
    "inventory_clean.csv",
    "calendar_clean.csv",
    "sales_calendar_clean.csv"
]:
    check_df = pd.read_csv(processed_dir / file_name)
    print(f"\n{file_name}")
    print("Shape:", check_df.shape)
    print("Missing values:", check_df.isnull().sum().sum())
    print("Duplicate rows:", check_df.duplicated().sum())



sales_clean.csv
Shape: (36550, 6)
Missing values: 0
Duplicate rows: 0

inventory_clean.csv
Shape: (4800, 8)
Missing values: 0
Duplicate rows: 0

calendar_clean.csv
Shape: (731, 11)
Missing values: 0
Duplicate rows: 0

sales_calendar_clean.csv
Shape: (36550, 16)
Missing values: 0
Duplicate rows: 0


## Key Findings and Conclusion

The data cleaning process covered the sales, inventory, and calendar datasets to prepare consistent and analysis-ready data for downstream RetailPulse modules.

**Key findings:**

- Sales dataset: 36,550 rows with 6 columns, containing 50 unique SKUs.
- Inventory dataset: 4,800 rows with 8 columns, covering 200 unique SKUs.
- Calendar dataset: 731 rows with 11 columns, covering January 2024 to December 2025.
- Missing values in the calendar's `holiday` and `promotion_event` columns were handled by filling them with `"No Holiday"` and `"No Promotion"`, respectively.
- Cleaned datasets were saved to the processed data folder as `sales_clean.csv`, `inventory_clean.csv`, and `calendar_clean.csv`.
- Sales and calendar datasets were merged using the date column to create `sales_calendar_clean.csv`.
- The merge was validated using a many-to-one relationship to help ensure calendar records were correctly matched to sales dates.

**Limitations:** This notebook focuses on data cleaning, missing-value handling, dataset integration, and validation. It does not train predictive models or evaluate forecasting performance. Non-positive sales records identified during exploration should be reviewed according to business rules rather than automatically removed without justification.

**Next step:** Use the cleaned and integrated datasets for demand forecasting, customer segmentation, churn analysis, inventory optimization, and Power BI dashboard development.